# 💧 LFM3-Audio - Teach the model a new voice with Transformers

This tutorial demonstrates how to fine-tune our LFM3 speech model [`LiquidAI/LFM3-Audio-2B`](https://huggingface.co/LiquidAI/LFM3-Audio-2B)
so it speaks with a **new, consistent voice**.

LFM3-Audio generates speech itself: for every 80 ms frame of audio it predicts 12 codebook tokens of the Liquid codec, alongside the text tokens.
Fine-tuning therefore trains **two losses at once**: the usual next-token loss on text, and an **audio loss** on the speech frames.
The model computes both for you; you only need to build the right batches.

We use [LJSpeech](https://huggingface.co/datasets/MikhailT/lj-speech), ~24 hours of a single speaker reading non-fiction books,
and teach the model to use this voice when asked for *"Linda's voice"*, a voice it has never heard of.

## 🎯 What You'll Find:
- **Data preparation** - turn (text, recording) pairs into training conversations with speech targets
- **LoRA + SFT** - parameter-efficient fine-tuning of the language model and the audio decoder with the 🤗 `Trainer`
- **Evaluation** - listen before/after, check the speech is still intelligible, and measure how close the voice is to the target speaker

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type` (an A100 is recommended; ~15 GB of GPU memory is used with the default settings)
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages:

In [ ]:
!pip install -qqq "transformers==5.20.2" "torchao==0.18.0" accelerate datasets peft librosa soundfile speechbrain --progress-bar off

Let's now verify the packages are installed correctly

In [ ]:
import os
import torch
import transformers
import peft
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"🧩 PEFT version: {peft.__version__}")

# Loading the model from Transformers 🤗

The **Liquid codec** converts between 24 kHz waveforms and the discrete speech codes the model reads and writes.
It lives in the `audio_tokenizer/` folder of the model repository; the processor loads it as `processor.audio_tokenizer`, and we move it to the GPU.

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Audio-2B"
device = "cuda"

print("🧠 Loading model...")
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map=device, trust_remote_code=True
)

print("📚 Loading processor and audio codec...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device)
tokenizer = processor.tokenizer

print("\n✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

## 🛠️ Helpers for speech in and out

`tts()` reads a sentence aloud with a given system prompt, and `asr()` transcribes a waveform.
We will use the model's own ASR to check that the fine-tuned voice stays intelligible.

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

SR = 24_000  # the codec works at 24 kHz


@torch.inference_mode()
def tts(model, text, system_prompt, seed=0):
    torch.manual_seed(seed)
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
        {"role": "assistant", "content": "<|audio_start|>"},
    ]
    inputs = processor.apply_chat_template(
        conversation, continue_final_message=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=600,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )
    codes = out.audio_codes[:1, :, : out.audio_lengths[0]]
    return codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()


@torch.inference_mode()
def asr(model, waveform):
    sf.write("_asr.wav", waveform, SR)
    inputs = processor.apply_chat_template(
        [
            {"role": "system", "content": "Perform ASR in English."},
            {"role": "user", "content": [{"type": "audio", "audio": "_asr.wav"}]},
        ],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(device)
    out = model.generate(**inputs, max_new_tokens=256)
    return processor.decode(out.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def play(waveform, sr=SR):
    display(Audio(waveform, rate=sr))

# 📚 Load the dataset

We use one of the 8 shards of [MikhailT/lj-speech](https://huggingface.co/datasets/MikhailT/lj-speech) (~1,600 clips, ~3 hours).
We keep the audio as raw bytes (`decode=False`) and decode it with `soundfile` ourselves.

In [ ]:
import io
from datasets import Audio as AudioFeature, load_dataset

N_TRAIN, N_EVAL = 1200, 20

# verification_mode="no_checks": we load one shard, which doesn't match the "full" split in the dataset card
raw_ds = load_dataset("MikhailT/lj-speech", data_files="data/full-00000-of-00008-*.parquet", split="train", verification_mode="no_checks")
raw_ds = raw_ds.cast_column("audio", AudioFeature(decode=False)).shuffle(seed=0).select(range(N_TRAIN + N_EVAL))

print("✅ Dataset loaded:", raw_ds)
sample = raw_ds[0]
waveform, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
print(f"\n📝 Single Sample: {sample['normalized_text']}")
play(waveform, sr)

## 🎧 Before fine-tuning

We will ask for **"Linda's voice"**. The model knows how to read in a female voice, but has no idea who Linda is,
so let's hear what it does before training. Keep these clips to compare with after training.

In [ ]:
SYSTEM_PROMPT = "Perform TTS in American English using Linda's voice."

test_sentences = [
    "The museum opens at nine o'clock and closes shortly after sunset.",
    "Please remember to bring your umbrella tomorrow.",
    "Our team shipped the new feature ahead of schedule.",
    "The river was calm, and the boats drifted slowly past the old bridge.",
    "Can you tell me where the nearest train station is?",
]

before = [tts(model, s, SYSTEM_PROMPT) for s in test_sentences]
for sentence, wav in zip(test_sentences[:2], before[:2]):
    print("🗣️", sentence)
    play(wav)

# 🔢 Encode the recordings into speech codes

The model learns to predict the **codec codes** of each recording, so we encode every clip once, up front, with
`processor.encode_audio()` (24 kHz mono in, a `(num_frames, 12)` array of codes out, one frame per 80 ms).
This takes a couple of minutes.

In [ ]:
import librosa


def encode(example):
    wav, sr = sf.read(io.BytesIO(example["audio"]["bytes"]), dtype="float32")
    wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    example["codes"] = processor.encode_audio(wav).tolist()
    return example


encoded_ds = raw_ds.map(encode, remove_columns=["audio"], desc="Encoding speech")
train_dataset = encoded_ds.select(range(N_TRAIN))
eval_dataset = encoded_ds.select(range(N_TRAIN, N_TRAIN + N_EVAL))

print("✅ Speech encoded:")
print(f"   📚 Train samples: {len(train_dataset)}")
print(f"   🧪 Eval samples: {len(eval_dataset)}")
print(f"   🎵 Codes of the first clip: {np.array(train_dataset[0]['codes']).shape}")

## Collate function

Each training example is a TTS conversation whose assistant turn is the speech itself:

```
<|im_start|>system
Perform TTS in American English using Linda's voice.<|im_end|>
<|im_start|>user
{text}<|im_end|>
<|im_start|>assistant
<|audio_start|><|audio_out|> × (num_frames + 1)<|im_end|>
```

Each `<|audio_out|>` placeholder stands for one speech frame; its embedding is replaced by the frame's codes (`audio_out_codes`).
The extra last frame is an **end-of-speech** frame, after which the model switches back to text and closes the turn.

The model takes two kinds of labels:
- `labels` (text): `-100` everywhere except `<|im_end|>` and the newline after it, so the model learns to end the turn after speaking
- `audio_labels` (speech): the codes of every frame, one row per `<|audio_out|>` placeholder, in order across the batch

`<|audio_start|>` is not trained: at inference we prefill it, as in the `tts()` helper above.

In [ ]:
AUDIO_OUT = processor.audio_out_token
EOS_CODE = processor.audio_eos_code


def create_collate_fn(system_prompt):
    def collate_fn(examples):
        input_ids, labels, codes = [], [], []
        for example in examples:
            prompt = processor.apply_chat_template(
                [{"role": "system", "content": system_prompt}, {"role": "user", "content": example["normalized_text"]}],
                add_generation_prompt=True, tokenize=False,
            )
            frames = torch.tensor(example["codes"])
            frames = torch.cat([frames, torch.full((1, frames.shape[1]), EOS_CODE)])  # end-of-speech frame
            prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            answer_ids = tokenizer("<|audio_start|>" + AUDIO_OUT * len(frames) + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
            answer_labels = [t if t != processor.audio_out_token_id else -100 for t in answer_ids]
            answer_labels[0] = -100  # <|audio_start|> is prefilled at inference
            input_ids.append(prompt_ids + answer_ids)
            labels.append([-100] * len(prompt_ids) + answer_labels)
            codes.append(frames)

        max_len = max(len(ids) for ids in input_ids)
        pad = lambda seq, value: seq + [value] * (max_len - len(seq))
        batch = {
            "input_ids": torch.tensor([pad(ids, tokenizer.pad_token_id) for ids in input_ids]),
            "attention_mask": torch.tensor([pad([1] * len(ids), 0) for ids in input_ids]),
            "labels": torch.tensor([pad(lab, -100) for lab in labels]),
            "audio_out_codes": torch.cat(codes),
        }
        batch["audio_labels"] = batch["audio_out_codes"].clone()
        return batch
    return collate_fn


collate_fn = create_collate_fn(SYSTEM_PROMPT)

batch = collate_fn([train_dataset[0], train_dataset[1]])
print({k: tuple(v.shape) for k, v in batch.items()})
with torch.no_grad():
    out = model(**{k: v.to(device) for k, v in batch.items()})
# the only supervised text tokens are <|im_end|> and the newline after it, which the model already predicts well: almost all the loss is on speech
print(f"📉 Loss before training: {out.loss.item():.3f} (text {out.text_loss.item():.3f}, audio {out.audio_loss.item():.3f})")

## Wrap the model with PEFT

We add LoRA adapters to the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), gated short convolutions (`in_proj`, `out_proj`) and MLP (`w1`, `w2`, `w3`) of the **language model** (which decides what to say and plans the speech)
and the **depth decoder** (the small transformer that turns each hidden state into the 12 codes of a speech frame).
The audio encoder, used only for speech *input*, stays frozen. LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full fine-tuning
- Lower → needs even less compute resources

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    # optional: add "|gate_proj" to the group for LFM3's per-head attention gate
    target_modules=r"model\.(language_model|depth_decoder)\..*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|w1|w2|w3)",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 🚀 Launch Training

We are now ready to launch an SFT run with the 🤗 `Trainer`. Our collate function builds the batches, so we keep all dataset columns
(`remove_unused_columns=False`). Two epochs over 1,200 clips take a few minutes on an A100.

> **Why not TRL's `SFTTrainer`?** It computes its own next-token loss on the text logits, which ignores the speech frames.
> The plain `Trainer` uses the loss returned by the model, which combines the text loss and the audio loss.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="lfm3-audio-linda",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
)

print("🏗️  Creating trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print("\n🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"💾 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# 🎧 After fine-tuning

Same sentences, same prompt, same seeds. Listen and compare with the clips above.

In [ ]:
model.eval()
after = [tts(model, s, SYSTEM_PROMPT) for s in test_sentences]
for sentence, wav_before, wav_after in zip(test_sentences, before, after):
    print("🗣️", sentence)
    print("   before:"); play(wav_before)
    print("   after:"); play(wav_after)

## ✅ Is it still intelligible?

A new voice is only useful if the words are still right. We transcribe the generated speech with the model's own ASR.

In [ ]:
for sentence, wav in zip(test_sentences, after):
    print(f"📝 {sentence}\n🎙️ {asr(model, wav)}\n")

## 📏 How close is the voice to the target speaker?

We embed each clip with a speaker-verification model ([ECAPA-TDNN](https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb))
and compute the cosine similarity to the average embedding of held-out LJSpeech recordings. Higher means closer to the target voice;
other recordings of the real speaker typically score around 0.84.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

speaker_encoder = EncoderClassifier.from_hparams("speechbrain/spkrec-ecapa-voxceleb", savedir="ecapa", run_opts={"device": device})


def speaker_embedding(wav, sr):
    wav = torch.tensor(librosa.resample(wav, orig_sr=sr, target_sr=16_000), dtype=torch.float32)[None]
    return torch.nn.functional.normalize(speaker_encoder.encode_batch(wav).squeeze().cpu(), dim=0)


reference_ds = load_dataset("MikhailT/lj-speech", data_files="data/full-00001-of-00008-*.parquet", split="train", verification_mode="no_checks")
reference_ds = reference_ds.cast_column("audio", AudioFeature(decode=False)).select(range(30))
reference = torch.stack([speaker_embedding(*sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="float32")) for r in reference_ds])
reference = torch.nn.functional.normalize(reference.mean(0), dim=0)

similarity = lambda clips: np.mean([float(speaker_embedding(w, SR) @ reference) for w in clips])
print(f"🎯 Speaker similarity to LJSpeech  before: {similarity(before):.2f}   after: {similarity(after):.2f}")

# 💾 Save the model

Save the LoRA adapter (a few tens of MB), or merge it into the base weights to obtain a regular checkpoint that loads exactly like the original model.

In [ ]:
model.save_pretrained("./lfm3-audio-linda-lora")
print("💾 Adapter saved to: ./lfm3-audio-linda-lora")

In [ ]:
print("🔄 Merging LoRA weights...")
merged = model.merge_and_unload()
merged.save_pretrained("./lfm3-audio-linda")
processor.save_pretrained("./lfm3-audio-linda")
print("💾 Model saved to: ./lfm3-audio-linda")

# 🚀 Next steps

- **Your own voice**: replace LJSpeech with recordings of the voice you want; a few hundred clips of a single speaker with accurate transcripts are enough to start. Keep the clips under ~20 seconds.
- **Several custom voices**: give each speaker its own name in the system prompt and mix them in the same training set.
- **Other tasks**: the same collate function pattern works for any task with speech output (e.g. speech-to-speech translation); for text-only outputs, drop `audio_out_codes`/`audio_labels` and put the audio in the user turn instead.